# EDA 01 — Kaggle Amazon Products (1.43M)
**Purpose:** Shape check and landscape scan of the Kaggle dataset using DuckDB.  
**Data:** `amazon_products.csv` (1,426,337 rows × 11 columns)  
**Engine:** DuckDB (direct CSV query, no loading step)

In [ ]:
import duckdb

con = duckdb.connect()

con.sql("""
    SELECT COUNT(*) as total_rows
    FROM read_csv_auto('../collection/amazon_products.csv')
""").show()

### Finding: Dataset Scale
1.43M products in the Kaggle dataset. DuckDB reads the 300 MB CSV directly — no upload, no cloud.

In [ ]:
con.sql("""
    SELECT column_name, column_type
    FROM (DESCRIBE SELECT * FROM read_csv_auto('../collection/amazon_products.csv'))
""").show()

### Schema
11 columns: asin, title, imgUrl, productURL, stars, reviews, price, listPrice, category_id, isBestSeller, boughtInLastMonth. All typed correctly by DuckDB auto-detection.

In [ ]:
con.sql("""
    SELECT 
        COUNT(*) as total,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) as null_price,
        SUM(CASE WHEN stars IS NULL THEN 1 ELSE 0 END) as null_stars,
        SUM(CASE WHEN reviews IS NULL THEN 1 ELSE 0 END) as null_reviews,
        SUM(CASE WHEN boughtInLastMonth IS NULL THEN 1 ELSE 0 END) as null_bought,
        SUM(CASE WHEN reviews = 0 THEN 1 ELSE 0 END) as zero_reviews,
        SUM(CASE WHEN boughtInLastMonth = 0 THEN 1 ELSE 0 END) as zero_bought,
        SUM(CASE WHEN price = 0 THEN 1 ELSE 0 END) as zero_price
    FROM read_csv_auto('../collection/amazon_products.csv')
""").show()

### Finding: Ghost Marketplace
Zero nulls across all columns — clean data. But 79% of products have zero reviews (1.13M) and 64% have zero monthly purchases (918K). The "active" marketplace is a small fraction of the total catalog.

In [ ]:
con.sql("""
    SELECT 
        ROUND(MIN(price), 2) as min_price,
        ROUND(MEDIAN(price), 2) as median_price,
        ROUND(AVG(price), 2) as avg_price,
        ROUND(MAX(price), 2) as max_price,
        ROUND(MIN(stars), 1) as min_stars,
        ROUND(AVG(stars), 1) as avg_stars,
        ROUND(MAX(stars), 1) as max_stars,
        MAX(reviews) as max_reviews,
        MAX(boughtInLastMonth) as max_bought,
        SUM(CASE WHEN isBestSeller THEN 1 ELSE 0 END) as bestseller_count
    FROM read_csv_auto('../collection/amazon_products.csv')
""").show()

### Finding: Ghost Marketplace
Zero nulls across all columns — clean data. But 79% of products have zero reviews (1.13M) and 64% have zero monthly purchases (918K). The "active" marketplace is a small fraction of the total catalog.

In [ ]:
con.sql("""
    SELECT 
        c.category_name,
        COUNT(*) as products,
        ROUND(AVG(p.price), 2) as avg_price,
        ROUND(AVG(p.stars), 2) as avg_stars,
        SUM(CASE WHEN p.reviews > 0 THEN 1 ELSE 0 END) as has_reviews,
        SUM(p.boughtInLastMonth) as total_bought,
        SUM(CASE WHEN p.isBestSeller THEN 1 ELSE 0 END) as bestsellers
    FROM read_csv_auto('../collection/amazon_products.csv') p
    JOIN read_csv_auto('../collection/amazon_categories.csv') c
        ON p.category_id = c.id
    GROUP BY c.category_name
    ORDER BY total_bought DESC
    LIMIT 20
""").show()

### Finding: Kitchen & Dining Dominance
Kitchen & Dining leads with 10.4M monthly buys from only 4,882 products — highest demand-per-product ratio. Beauty/personal care subcategories dominate the top 20. Notably, `has_reviews` shows 0 for most top categories despite having sales — worth investigating.

In [ ]:
con.sql("""
    SELECT 
        COUNT(DISTINCT c.category_name) as total_categories,
        SUM(p.boughtInLastMonth) as total_bought_all
    FROM read_csv_auto('../collection/amazon_products.csv') p
    JOIN read_csv_auto('../collection/amazon_categories.csv') c
        ON p.category_id = c.id
""").show()

### Finding: 248 Subcategories, 202M Monthly Purchases
248 Kaggle subcategories (vs McAuley's 33 top-level categories — a granularity mismatch to reconcile in Silver). 202M total monthly purchases across the entire catalog.

In [ ]:
con.sql("""
    SELECT 
        c.category_name,
        COUNT(*) as products,
        ROUND(AVG(p.price), 2) as avg_price,
        SUM(p.boughtInLastMonth) as total_bought,
        ROUND(100.0 * SUM(CASE WHEN p.reviews > 0 THEN 1 ELSE 0 END) / COUNT(*), 1) as pct_with_reviews
    FROM read_csv_auto('../collection/amazon_products.csv') p
    JOIN read_csv_auto('../collection/amazon_categories.csv') c
        ON p.category_id = c.id
    GROUP BY c.category_name
    ORDER BY total_bought ASC
    LIMIT 20
""").show()

### Finding: Dead Zones
Bottom 20 categories have near-zero monthly purchases. Computer Servers ($1,534 avg price), Smart Home subcategories, and legacy gaming (PSP, Nintendo DS) are graveyards. High-price categories (Servers, Computers & Tablets at $544) have catalog but no volume — likely B2B or niche. Gift Cards showing zero bought is a data artifact (purchased differently). These are categories to flag as "avoid" in the Category Scout tool.

In [ ]:
con.sql("""
    SELECT 
        CASE 
            WHEN price < 10 THEN '1. Under $10'
            WHEN price < 25 THEN '2. $10-25'
            WHEN price < 50 THEN '3. $25-50'
            WHEN price < 100 THEN '4. $50-100'
            ELSE '5. $100+'
        END as price_tier,
        COUNT(*) as products,
        ROUND(AVG(boughtInLastMonth), 1) as avg_bought,
        ROUND(AVG(stars), 2) as avg_stars,
        ROUND(100.0 * SUM(CASE WHEN isBestSeller THEN 1 ELSE 0 END) / COUNT(*), 2) as pct_bestseller
    FROM read_csv_auto('../collection/amazon_products.csv')
    WHERE price > 0
    GROUP BY 1
    ORDER BY 1
""").show()

### Finding: Cheap Wins Volume — But Is It Universal?
Overall, budget products dominate: Under $10 averages 219 bought/month vs $100+ at just 36. Ratings and bestseller % also decline with price. But this is the AGGREGATE picture — the thesis says this flips in certain categories. Category-level price analysis (in Silver/Gold) will test whether premium wins in trust categories like Health, Baby, Electronics.

In [ ]:
con.sql("""
    SELECT 
        CASE 
            WHEN boughtInLastMonth = 0 THEN '1. Zero sales'
            WHEN boughtInLastMonth < 50 THEN '2. 1-49'
            WHEN boughtInLastMonth < 500 THEN '3. 50-499'
            WHEN boughtInLastMonth < 5000 THEN '4. 500-4999'
            ELSE '5. 5000+'
        END as sales_tier,
        COUNT(*) as products,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) as pct_of_catalog,
        SUM(boughtInLastMonth) as total_bought,
        ROUND(100.0 * SUM(boughtInLastMonth) / SUM(SUM(boughtInLastMonth)) OVER (), 1) as pct_of_revenue
    FROM read_csv_auto('../collection/amazon_products.csv')
    GROUP BY 1
    ORDER BY 1
""").show()

### Finding: Power Law Marketplace
64% of products have zero sales. But 0.4% of products (5,548) generate 25% of all revenue. The middle tier (500-4,999 bought/month, 6.4% of catalog) drives the most volume at 48%. Notably, the 1-49 tier is EMPTY — Amazon's boughtInLastMonth field rounds small values to zero, creating a data cliff. This means the "zero sales" group likely contains products with <50 monthly sales — the ghost marketplace is smaller than it appears.

## Summary
- **1.43M products**, 248 subcategories, 202M total monthly purchases
- **Ghost marketplace:** 64% zero sales, 79% zero reviews — active catalog is a fraction
- **Price skew:** Median $20, mean $43, max $19.7K
- **Cheap wins volume overall** but thesis predicts category-level reversal
- **Power law:** 0.4% of products drive 25% of revenue; 6.4% drive 48%
- **Data cliff at 50:** Amazon rounds boughtInLastMonth — no values between 1-49
- **Dead zones:** Computer Servers, Smart Home, legacy gaming consoles
- **Hot zones:** Kitchen & Dining (10.4M bought, only 4.9K products)

### Next: EDA 02 — McAuley Metadata (35M products)